In [ ]:
#################################################################
# 모델 8: Gradient Boosting (RandomizedSearchCV)
#################################################################
import numpy as np
from time import time
import joblib
import warnings
from sklearn.exceptions import ConvergenceWarning

# 모듈 임포트
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier # (Scikit-learn의 GBRT) [cite: 2065]
from scipy.stats import uniform, randint

# 경고 메시지 무시
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module='sklearn')

print("--- 0. 공통 모듈 임포트 완료 ---")

--- 0. 공통 모듈 임포트 완료 ---


In [ ]:
# --- 1. Original MNIST 데이터셋 로딩 (70,000개) ---
print("\n--- 1. Original MNIST 데이터셋 로딩 (70,000개) ---")
start_time = time()
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X = mnist.data/255
y = mnist.target.astype(np.uint8)
print(f"   ...로딩 완료. (소요 시간: {time() - start_time:.2f}초)")

print("\n--- 모델 8: Gradient Boosting ---")


--- 1. Original MNIST 데이터셋 로딩 (70,000개) ---
   ...로딩 완료. (소요 시간: 7.04초)

--- 모델 8: Gradient Boosting ---


In [ ]:
# --- 8-1. 스케일링 테스트 ---
print("   8-1. 스케일링 테스트: 불필요 (트리 기반 앙상블)")

   8-1. 스케일링 테스트: 불필요 (트리 기반 앙상블)


In [ ]:
# --- 8-2. 하이퍼파라미터 튜닝 (RandomizedSearchCV, 전체 X, y 사용) ---
# [cite_start]Gradient Boosting은 튜닝 시간이 매우 오래 걸리므로 RandomizedSearchCV를 사용합니다. [cite: 5330]
print("   8-2. RandomizedSearchCV 튜닝 시작 (전체 데이터 사용, 시간이 매우 오래 걸릴 수 있음)...")
pipeline = Pipeline([
    ('model', GradientBoostingClassifier(random_state=42))
])

# [cite_start]max_depth=2, n_estimators=3/500, learning_rate=1.0/0.1/0.05 등 예시 참고 [cite: 2141, 2151, 2192]
param_distribs = {
    'model__n_estimators': randint(100, 300), # 트리 개수 [cite: 2192]
    'model__learning_rate': uniform(0.01, 0.2), # 학습률 [cite: 2151]
    'model__max_depth': randint(3, 7) # 트리 깊이 [cite: 2141]
}

# [cite_start]n_iter=10: 10개 조합만 랜덤하게 테스트 [cite: 5337]
rnd_search_gb = RandomizedSearchCV(pipeline, param_distribs, n_iter=10, cv=3, scoring='accuracy', n_jobs=-1, verbose=2, random_state=42)
start_time = time()
rnd_search_gb.fit(X, y)
print(f"   ...튜닝 완료. (소요 시간: {time() - start_time:.2f}초)")
print(f"   - 최적 파라미터: {rnd_search_gb.best_params_}")
print(f"   - 최고 정확도: {rnd_search_gb.best_score_:.4f}")

   8-2. RandomizedSearchCV 튜닝 시작 (전체 데이터 사용, 시간이 매우 오래 걸릴 수 있음)...
Fitting 3 folds for each of 10 candidates, totalling 30 fits


In [ ]:
# --- 8-3. 최적 모델 저장 ---
model_filename = '08_gradient_boosting_best_0-1.joblib'
joblib.dump(rnd_search_gb.best_estimator_, model_filename)
print(f"   ...최적 모델을 '{model_filename}' 파일로 저장했습니다.")